In [6]:
%pip install -q --no-cache-dir --upgrade \
    transformers==4.53.3 \
    trl==0.19.1 \
    peft==0.17.1 \
    accelerate==1.8.1 \
    bitsandbytes==0.49.2 \
    datasets==3.6.0 \
    scikit-learn==1.7.2 \
    sentencepiece==0.2.0

In [7]:

import numpy as np
import pandas as pd
import torch
import os
import random
from trl import *
from peft import *
import zipfile
from pathlib import Path
from transformers import *
from scipy.sparse import hstack
from datasets import Dataset

seed = 42
model_id = "Qwen/Qwen3-4B-Instruct-2507"

MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 256
MAX_COMPLETION_LENGTH = 768

out = Path("/content/outputs")
sft_d = out / "sft"
dp_d = out / "dpo"
rwm_d = out / "rwd_m"
dpo_qd = out / "dpo_q"
simp_d = out / "simpo"

os.environ["PYTHONHASHSEED"] = str(seed)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
set_seed(seed)

out.mkdir(parents=True, exist_ok=True)

In [8]:
from pathlib import Path
import json
import zipfile

archive_path = Path(
    "/content/ml-olympiad-red-task-c1005bf0-8695-451a-9616-87c8687dce27.zip"
)

with zipfile.ZipFile(archive_path) as archive:
    archive.extractall("/content")

data_dir = Path("/content/ml-olympiad-red-task/data")


def read_jsonl(filename):
    with open(data_dir / filename, encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


kid_adult = read_jsonl("kid_adult.jsonl")
good_bad = read_jsonl("good_bad.jsonl")
public_style = read_jsonl("public_test_style.jsonl")
public_quality = read_jsonl("public_test_quality.jsonl")


In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


def user(text):
    return [{"role": "user", "content": text}]


def assistant(text):
    return [{"role": "assistant", "content": text}]


sft_da = Dataset.from_list([
    {
        "prompt": user(row["prompt"]),
        "completion": assistant(row["kid"]),
    }
    for row in kid_adult
])

sty_da = Dataset.from_list([
    {
        "prompt": user(row["prompt"]),
        "chosen": assistant(row["kid"]),
        "rejected": assistant(row["adult"]),
    }
    for row in kid_adult
])

re_da = Dataset.from_list([
    {
        "chosen": [
            {"role": "user", "content": row["instruction"]},
            {"role": "assistant", "content": row["chosen"]},
        ],
        "rejected": [
            {"role": "user", "content": row["instruction"]},
            {"role": "assistant", "content": row["rejected"]},
        ],
    }
    for row in good_bad
])

qu_da = Dataset.from_list([
    {
        "prompt": user(row["instruction"]),
        "chosen": assistant(row["chosen"]),
        "rejected": assistant(row["rejected"]),
    }
    for row in good_bad
])

print(sft_da)
print(sty_da)
print(re_da)
print(qu_da)

loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1489
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 1489
})
Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 2226
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 2226
})


In [10]:
import joblib

style_f = joblib.load(
    "/content/ml-olympiad-red-task/metrics/style_clf.pkl"
)
style_clf = style_f["clf"]
style_vecs = style_f["vecs"]

simp_cl = int(
    np.where(style_clf.classes_ == 1)[0][0]
)


def p_simp(texts):
    ft = hstack([
        vectorizer.transform(texts)
        for vectorizer in style_vecs
    ])

    prb = style_clf.predict_proba(ft)

    return float(
        prb[:, simp_cl].mean()
    )


print(
    "kid:",
    p_simp([row["kid"] for row in kid_adult])
)

print(
    "adult:",
    p_simp([row["adult"] for row in kid_adult])
)

kid: 0.9744881038704017
adult: 0.02487847940706435


In [23]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

targets = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

causal_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=targets,
)

reward_lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=targets,
    modules_to_save=["score"],
)

common_args = {
    "seed": seed,
    "data_seed": seed,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "fp16": True,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {
        "use_reentrant": False,
    },
    "optim": "paged_adamw_8bit",
    "warmup_ratio": 0.05,
    "logging_steps": 10,
    "save_strategy": "no",
    "report_to": "none",
}


def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def base_model():
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False

    return prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )


def reward_base():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=1,
        problem_type="regression",
        quantization_config=bnb,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False

    return prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )


def adapter_model(folder, reference=False):
    model = PeftModel.from_pretrained(
        base_model(),
        folder,
        adapter_name="default",
        is_trainable=True,
    )

    if reference:
        model.load_adapter(
            folder,
            adapter_name="reference",
            is_trainable=False,
        )

    model.set_adapter("default")
    return model

In [17]:
@torch.inference_mode()
def style_metric(model):
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"

    questions = [
        row["prompt"]
        for row in public_style
    ]

    answers = []

    for start in range(0, len(questions), 2):
        batch = questions[start:start + 2]

        texts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": question}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for question in batch
        ]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LENGTH,
        ).to(model.device)

        prompt_length = inputs["input_ids"].shape[1]

        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        answers += tokenizer.batch_decode(
            output[:, prompt_length:],
            skip_special_tokens=True,
        )

        print(min(start + 2, len(questions)), "/", len(questions))

    tokenizer.padding_side = "right"

    return p_simp(answers)


def answer_logprob(model, question, answer):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=True,
        add_generation_prompt=True,
    )

    full = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ],
        tokenize=True,
        add_generation_prompt=False,
    )

    answer_ids = full[len(prompt):]
    prompt = prompt[-MAX_PROMPT_LENGTH:]
    answer_ids = answer_ids[:MAX_COMPLETION_LENGTH]

    ids = torch.tensor(
        [prompt + answer_ids],
        device=model.device,
    )

    labels = torch.tensor(
        [[-100] * len(prompt) + answer_ids],
        device=model.device,
    )

    with torch.no_grad():
        logits = model(
            input_ids=ids,
            use_cache=False,
        ).logits[:, :-1].float()

    labels = labels[:, 1:]
    mask = labels != -100

    safe_labels = labels.clone()
    safe_labels[~mask] = 0

    logprobs = torch.log_softmax(logits, dim=-1)

    selected = logprobs.gather(
        -1,
        safe_labels.unsqueeze(-1),
    ).squeeze(-1)

    return float(selected[mask].mean())


def policy_accuracy(model):
    correct = 0

    for i, row in enumerate(public_quality):
        good = answer_logprob(
            model,
            row["prompt"],
            row["chosen"],
        )

        bad = answer_logprob(
            model,
            row["prompt"],
            row["rejected"],
        )

        correct += int(good > bad)
        print(i + 1, "/", len(public_quality))

    return correct / len(public_quality)


def reward_score(model, question, answer):
    text = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        return float(
            model(**inputs).logits.reshape(-1)[0]
        )


def reward_accuracy(model):
    correct = 0

    for i, row in enumerate(public_quality):
        good = reward_score(
            model,
            row["prompt"],
            row["chosen"],
        )

        bad = reward_score(
            model,
            row["prompt"],
            row["rejected"],
        )

        correct += int(good > bad)
        print(i + 1, "/", len(public_quality))

    return correct / len(public_quality)

In [13]:
#первая таска (сфт)
import gc
clear_gpu()

model = get_peft_model(
    base_model(),
    causal_lora,
)

args = SFTConfig(
    output_dir=str(sft_d),
    learning_rate=2e-4,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    eos_token=tokenizer.eos_token,
    pad_token=tokenizer.pad_token,
    **common_args,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=sft_da,
    processing_class=tokenizer,
)

trainer.train()
model.save_pretrained(sft_d)

task1 = style_metric(model)
print(task1)

del trainer, model
clear_gpu()

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2560,
  "initializer_range": 0.02,
  "intermediate_size": 9728,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "ful

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
All model checkpoint weights were used when initializing Qwen3ForCausalLM.

All the weights of Qwen3ForCausalLM were initialized from the model checkpoint at Qwen/Qwen3-4B-Instruct-2507.
If your task is similar to the task the model of the checkpoint was trained on, you can already use Qwen3ForCausalLM for predictions without further training.
loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "temperature": 0.7,
  "top_k": 20,
  "to

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
The following columns in the Training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: completion, prompt. If completion, prompt are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
skipped Embedding(151936, 2560): 370.9375M params
skipped: 370.9375M params
***** Running training *****
  Num examples = 1,489
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 187
  Number of trainable parameters = 33,030,144


Step,Training Loss
10,2.409400
20,1.410600
30,1.311700
40,1.298300
50,1.305800
60,1.252200
70,1.222800
80,1.220000
90,1.229700
100,1.221400




Training completed. Do not forget to share your model on huggingface.co/models =)


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2560,
  "initializer_range": 0.02,
  "intermediate_size": 9728,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_

NameError: name 'style_metric' is not defined

In [20]:
# до этого сверху ещё была доп ячейка потому что у меня в ячейке сверху как можно видеть была ошибка небольшая, так что я сделал task1 = style_metric(model)
#print(task1) после этого, но удалил чтобы можно было сразу полностью весь пайплайн нормально запустить
# вот если что результат который выдал сфт: 0.9812567002968672
#вторая таска (дпо)
model = adapter_model(sft_d, reference=True)

args = DPOConfig(
    output_dir=str(dp_d),
    learning_rate=1e-5,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    beta=0.1,
    loss_type="sigmoid",
    model_adapter_name="default",
    ref_adapter_name="reference",
    **common_args,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=args,
    train_dataset=sty_da,
    processing_class=tokenizer,
)

trainer.train()

model.set_adapter("default")
model.save_pretrained(
    dp_d,
    selected_adapters=["default"],
)

task2 = style_metric(model)
print(task2)

del trainer, model
clear_gpu()

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2560,
  "initializer_range": 0.02,
  "intermediate_size": 9728,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "ful

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
All model checkpoint weights were used when initializing Qwen3ForCausalLM.

All the weights of Qwen3ForCausalLM were initialized from the model checkpoint at Qwen/Qwen3-4B-Instruct-2507.
If your task is similar to the task the model of the checkpoint was trained on, you can already use Qwen3ForCausalLM for predictions without further training.
loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "temperature": 0.7,
  "top_k": 20,
  "to

Extracting prompt in train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

...

44 / 50


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k'].
- `temperature`: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
- `top_p`: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
- `top_k`: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
If you're using a pretrained model, note that some of these attributes may be set through the model's `generation_config.json` file.


46 / 50


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k'].
- `temperature`: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
- `top_p`: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
- `top_k`: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
If you're using a pretrained model, note that some of these attributes may be set through the model's `generation_config.json` file.


48 / 50
50 / 50
0.9959648774626446


In [24]:
model = get_peft_model(
    reward_base(),
    reward_lora,
)

args = RewardConfig(
    output_dir=str(rwm_d),
    learning_rate=1e-4,
    max_length=MAX_LENGTH,
    remove_unused_columns=False,
    **common_args,
)

trainer = RewardTrainer(
    model=model,
    args=args,
    train_dataset=re_da,
    processing_class=tokenizer,
)

trainer.train()
model.save_pretrained(rwm_d)

task3 = reward_accuracy(model)
print(task3)

del trainer, model
clear_gpu()

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2560,
  "id2label": {
    "0": "LABEL_0"
  },
  "initializer_range": 0.02,
  "intermediate_size": 9728,
  "label2id": {
    "LABEL_0": 0
  },
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attentio

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
All model checkpoint weights were used when initializing Qwen3ForSequenceClassification.

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-4B-Instruct-2507 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
PyTorch: setting up devices
average_tokens_across_devices is True but world size is 1. Setting it to False automatically.


Map:   0%|          | 0/2226 [00:00<?, ? examples/s]

Map:   0%|          | 0/2226 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2226 [00:00<?, ? examples/s]

Using auto half precision backend
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
skipped Embedding(151936, 2560): 370.9375M params
skipped: 370.9375M params
***** Running training *****
  Num examples = 2,226
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 279
  Number of trainable parameters = 33,032,704
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
10,1.002600
20,0.185100
30,0.206400
40,0.010500
50,0.000200
60,0.051200
70,0.234000
80,0.073800
90,0.006900
100,0.010800




Training completed. Do not forget to share your model on huggingface.co/models =)


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2560,
  "initializer_range": 0.02,
  "intermediate_size": 9728,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_

1 / 50
2 / 50
3 / 50
4 / 50
5 / 50
6 / 50
7 / 50
8 / 50
9 / 50
10 / 50
11 / 50
12 / 50
13 / 50
14 / 50
15 / 50
16 / 50
17 / 50
18 / 50
19 / 50
20 / 50
21 / 50
22 / 50
23 / 50
24 / 50
25 / 50
26 / 50
27 / 50
28 / 50
29 / 50
30 / 50
31 / 50
32 / 50
33 / 50
34 / 50
35 / 50
36 / 50
37 / 50
38 / 50
39 / 50
40 / 50
41 / 50
42 / 50
43 / 50
44 / 50
45 / 50
46 / 50
47 / 50
48 / 50
49 / 50
50 / 50
Задача 3, accuracy = 1.0
